<a href="https://colab.research.google.com/github/YOUR-USERNAME/bags-vectors-transformers/blob/main/day1/notebooks/2_classic_cta_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bags, Vectors & Transformers
## Day 1 — Classic Text Analysis & the Document-Term Matrix  ·  **SOLUTIONS**

**A Methods Workshop in Computational Text Analysis**
Denise J. Roth · Strategic Communication Group · Wageningen University & Research

---

This is the **solutions** version. Every **✏️ Exercise** is filled in with one possible
answer. There is usually more than one correct way — if your approach differs but gives
sensible results, that is fine.


## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
from nltk.corpus import inaugural, stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import CountVectorizer

nltk.download("inaugural")
nltk.download("stopwords")
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("wordnet")

print("Setup complete!")

## 1. Loading a real corpus

In [ ]:
file_ids = inaugural.fileids()

print("Number of addresses:", len(file_ids))
print("First:", file_ids[0])
print("Last: ", file_ids[-1])
print()
print("A few in the middle:")
for fid in file_ids[28:32]:
    print("  ", fid)

In [ ]:
records = []
for fid in file_ids:
    year = int(fid[:4])
    president = fid[5:-4]
    text = inaugural.raw(fid)
    records.append({"year": year, "president": president, "text": text})

df = pd.DataFrame(records)
print("DataFrame shape:", df.shape)
df.head()

> **✏️ Exercise 1**
>
> Using the DataFrame `df`, print the **year** and **president** of the earliest and the
> most recent address.


In [ ]:
# ✅ Solution
earliest = df.loc[df["year"].idxmin()]
latest = df.loc[df["year"].idxmax()]

print(f"Earliest: {earliest['president']} ({earliest['year']})")
print(f"Latest:   {latest['president']} ({latest['year']})")

# Equivalent using sorting / iloc:
# df_sorted = df.sort_values("year")
# print(df_sorted.iloc[0][["year", "president"]])
# print(df_sorted.iloc[-1][["year", "president"]])

## 2. A first look at the texts

In [ ]:
df["n_words"] = df["text"].apply(lambda t: len(t.split()))

print(df[["year", "president", "n_words"]].head(10))
print()
print("Shortest speech:")
print(df.loc[df["n_words"].idxmin(), ["year", "president", "n_words"]])
print()
print("Longest speech:")
print(df.loc[df["n_words"].idxmax(), ["year", "president", "n_words"]])

In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(df["year"], df["n_words"], marker="o", color="#34B233")
plt.title("Length of inaugural addresses over time")
plt.xlabel("Year")
plt.ylabel("Number of words")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

> **✏️ Exercise 2**
>
> Do longer speeches cluster in a particular era? Write a sentence describing what you
> notice, then find the president who gave the longest address.


In [ ]:
# ✅ Solution
longest = df.loc[df["n_words"].idxmax()]
print(f"Longest address: {longest['president']} ({longest['year']}) "
      f"with {longest['n_words']} words.")

# Observation (one possible reading):
# The very longest speeches tend to sit in the 19th century (e.g. W. H. Harrison, 1841).
# Modern addresses are generally shorter and more uniform in length — arguably shaped by
# radio and television, which reward brevity.

## 3. Preprocessing (again!)

In [ ]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    """Clean a single document and return a space-joined string of tokens."""
    text = text.lower()
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t.isalpha()]
    tokens = [t for t in tokens if t not in stop_words]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return " ".join(tokens)

obama = df.loc[df["president"] == "Obama", "text"].iloc[0]
print("RAW:\n", obama[:200])
print("\nCLEANED:\n", preprocess(obama[:200]))

In [ ]:
df["clean"] = df["text"].apply(preprocess)
print(df.loc[0, "clean"][:300], "...")

> **✏️ Exercise 3**
>
> Add an argument `remove_stopwords=True` so you can turn stopword removal on and off.
> Test it on Obama's speech with stopwords **kept**.


In [ ]:
# ✅ Solution
def preprocess2(text, remove_stopwords=True):
    """Clean text, with optional stopword removal."""
    text = text.lower()
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t.isalpha()]
    if remove_stopwords:
        tokens = [t for t in tokens if t not in stop_words]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return " ".join(tokens)

print("Stopwords REMOVED:\n", preprocess2(obama[:200], remove_stopwords=True))
print("\nStopwords KEPT:\n", preprocess2(obama[:200], remove_stopwords=False))

## 4. Building the Document-Term Matrix

In [ ]:
vectorizer = CountVectorizer(min_df=2)
dtm = vectorizer.fit_transform(df["clean"])

print("DTM shape (documents x terms):", dtm.shape)
print("That is", dtm.shape[0], "speeches and", dtm.shape[1], "unique terms.")

In [ ]:
n_cells = dtm.shape[0] * dtm.shape[1]
n_nonzero = dtm.nnz
print(f"Total cells:      {n_cells:,}")
print(f"Non-zero cells:   {n_nonzero:,}")
print(f"Percentage zeros: {100 * (1 - n_nonzero / n_cells):.1f}%")

In [ ]:
dtm_df = pd.DataFrame(
    dtm.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=df["president"] + " (" + df["year"].astype(str) + ")",
)

print("DTM as a table (first 5 docs, a few columns):")
dtm_df.iloc[:5, :8]

> **✏️ Exercise 4**
>
> Look up how many times the word **"freedom"** appears in each speech. Which president
> used it most?


In [ ]:
# ✅ Solution
freedom = dtm_df["freedom"].sort_values(ascending=False)
print("Top 5 speeches by use of 'freedom':")
print(freedom.head(5))
print()
print("Most:", freedom.idxmax(), "->", freedom.max(), "times")

## 5. What can we do with a DTM?

In [ ]:
total_counts = dtm_df.sum(axis=0).sort_values(ascending=False)

print("Top 15 words across all inaugural addresses:")
print(total_counts.head(15))

In [ ]:
top15 = total_counts.head(15)

plt.figure(figsize=(11, 5))
plt.bar(top15.index, top15.values, color="#34B233")
plt.title("Most frequent words across all inaugural addresses")
plt.xlabel("Word")
plt.ylabel("Total frequency")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

> **✏️ Exercise 5**
>
> Pick **two** words you find interesting and make a bar chart comparing just those two.


In [ ]:
# ✅ Solution
words_to_compare = ["war", "peace"]
values = [dtm_df[w].sum() for w in words_to_compare]

plt.figure(figsize=(6, 5))
plt.bar(words_to_compare, values, color=["#1A1A2E", "#34B233"])
plt.title("'war' vs 'peace' across all inaugural addresses")
plt.ylabel("Total frequency")
plt.tight_layout()
plt.show()

for w, v in zip(words_to_compare, values):
    print(f"{w}: {v}")

## 6. Tracking a word over time

In [ ]:
def plot_word_over_time(word):
    if word not in dtm_df.columns:
        print(f"'{word}' is not in the vocabulary (maybe too rare, or removed in cleaning).")
        return
    counts = dtm_df[word].values
    plt.figure(figsize=(11, 5))
    plt.plot(df["year"], counts, marker="o", color="#1A1A2E")
    plt.title(f"Use of '{word}' in inaugural addresses over time")
    plt.xlabel("Year")
    plt.ylabel(f"Count of '{word}'")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_word_over_time("freedom")

In [ ]:
plot_word_over_time("war")

> **✏️ Exercise 6**
>
> Explore a few words of your choice: `"america"`, `"god"`, `"economy"`, or anything with
> a likely historical pattern. Remember to search for the **lemma**.


In [ ]:
# ✅ Solution
for w in ["america", "god", "economy"]:
    plot_word_over_time(w)

# Observations you might make:
# - "america" rises noticeably in modern speeches.
# - "god" appears throughout but is far from constant.
# - "economy" is largely a 20th/21st-century preoccupation — barely present early on.

## 7. Bonus: comparing two speeches

In [ ]:
def compare_speeches(pres1, pres2, n=10):
    row1 = dtm_df.loc[dtm_df.index.str.startswith(pres1)].iloc[0]
    row2 = dtm_df.loc[dtm_df.index.str.startswith(pres2)].iloc[0]
    diff = (row1 - row2).sort_values(ascending=False)
    print(f"Words more used by {pres1}:")
    print(diff.head(n))
    print(f"\nWords more used by {pres2}:")
    print(diff.tail(n)[::-1])

compare_speeches("Lincoln", "Trump")

> **✏️ Exercise 7**
>
> Compare two presidents of your choice.


In [ ]:
# ✅ Solution
compare_speeches("Kennedy", "Reagan")

# Reading the output: the top block are words Kennedy used more than Reagan
# in their (first) inaugural; the bottom block are words Reagan used more.
# This is a raw-count comparison — TF-IDF (next session) gives a more principled
# notion of "distinctive".

## Wrap-up

That's the full solution set. Remember these are *one* way to solve each exercise.

### Optional challenge

Normalize each row of the DTM so it sums to 1 (counts → proportions), then redo the
"freedom over time" plot. Does the picture change?


In [ ]:
# ✅ Solution
# Divide each row by its row total -> proportions
dtm_prop = dtm_df.div(dtm_df.sum(axis=1), axis=0)

plt.figure(figsize=(11, 5))
plt.plot(df["year"], dtm_prop["freedom"].values, marker="o", color="#34B233")
plt.title("Proportional use of 'freedom' over time (normalized)")
plt.xlabel("Year")
plt.ylabel("Share of words that are 'freedom'")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Why it matters: raw counts partly reflect how LONG a speech is. Normalizing to
# proportions controls for speech length, so spikes reflect genuine emphasis rather
# than just a wordier president. The shape of the "freedom" trend can shift noticeably.